In [33]:
import pickle
import numpy as np

In [34]:
# Load preprocessed dataset
with open("heart_disease_processed.pkl", "rb") as f:
    X_train, X_test, y_train, y_test, scaler = pickle.load(f)

In [35]:
# Convert dataset series/dataframes to numpy arrays
X_tr = X_train.to_numpy()
y_tr = y_train.to_numpy()
X_te = X_test.to_numpy()
y_te = y_test.to_numpy()

In [124]:
# Utils
def gini(y): # y is labels
    labels_count = len(y)
    if labels_count == 0:
        return 0
    _, frequencies = np.unique(y, return_counts=True)
    probabilities = np.sum((frequencies / labels_count) ** 2)
    return 1 - probabilities

def split_labels(y, feature, threshold):
    y_left = y[feature <= threshold]
    y_right = y[feature > threshold]
    return y_left, y_right

def weighted_gini(y_left, y_right):
    len_left = len(y_left)
    len_right = len(y_right)
    len_total = len_left + len_right
    return (len_left * gini(y_left) + len_right * gini(y_right)) / len_total

def find_best_split(X, y, num_thresholds = 10):
    lowest_gini = np.inf
    lowest_gini_feature_thresh = ()

    for feature_idx in range(X.shape[1]):

        unique_vals = np.unique(X[:, feature_idx])
        if len(unique_vals) > num_thresholds:
            thresholds = np.linspace(unique_vals.min(), unique_vals.max(), num_thresholds)
        else:
            thresholds = unique_vals

        for threshold in thresholds:
            y_left, y_right = split_labels(y, X[:,feature_idx], threshold)
            current_gini = weighted_gini(y_left, y_right)

            if current_gini < lowest_gini:
                lowest_gini = current_gini
                lowest_gini_feature_thresh = (feature_idx, threshold)
    return lowest_gini, *lowest_gini_feature_thresh

In [125]:
# Test gini
gini(y_train)

np.float64(0.4950823031213716)

In [126]:
# Test split_labels and weighted gini
age_idx = X_train.columns.get_loc("age")

y_left, y_right = split_labels(y_tr, X_tr[:, age_idx], 0.1)

weighted_gini(y_left, y_right)

np.float64(0.45789222047674333)

In [127]:
# Test find_best_split
find_best_split(X_tr, y_tr)

(np.float64(0.3642147663558955), 15, np.float64(0.0))